# Acoustic-Aware Fusion — severity LOSO training (full UA-Speech M6, 15 folds)

In [ ]:
# -----------------------------------------------------------------------------
# 1.1  Imports and project root
# -----------------------------------------------------------------------------
import time
SESSION_START = time.monotonic()

import json
import os
import platform
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# The notebook may be started from notebooks/ or from the repo root: walk up
# until the directory holding src/config.py is found.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src import feature_store
from src.console import (print_banner, print_header, print_kv, print_note, print_status,
                         print_table, silence_library_progress)
from src.training.data import load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model, parameter_counts
from src.training.reporting import (check_frozen_config_guard, print_feature_audit,
                                    print_final_run_configuration, summarize_registry,
                                    write_frozen_config)
from src.training.runner import TrainingConfig, run_training
from src.training.session import project_runtime, subset_blocks
from src.training.utils import (configure_local_runtime, memory_status, resolve_amp_dtype,
                                resolve_device)
from src.vad_cache import verify_vad_span_cache

config.ensure_directories()
silence_library_progress()     # no per-parameter "Loading weights" bars on every model build

# src.training.reporting selects matplotlib's file-only "Agg" backend at import
# (it writes figures to outputs/ from worker-free scripts too); switch this
# kernel back to inline so the figures in Section 7 are displayed.
%matplotlib inline
print_status(f"Project root: {PROJECT_ROOT}", ok=True)

In [ ]:
# -----------------------------------------------------------------------------
# 1.2  Hardware and local runtime profile
# -----------------------------------------------------------------------------
import psutil

device = resolve_device(None)
if device.type != "cuda":
    raise RuntimeError("CUDA is not available to this kernel — select the Python environment "
                       "with the CUDA build of torch (e.g. the 'torch-gpu' conda env).")
configure_local_runtime(device)          # cudnn.benchmark + VRAM cap sized to free GPU memory

props = torch.cuda.get_device_properties(device)
gpu_free, gpu_total = torch.cuda.mem_get_info(device)
memory = memory_status()
disk = shutil.disk_usage(PROJECT_ROOT)

print_header("Environment")
print_kv("Python / PyTorch / CUDA", f"{platform.python_version()} / {torch.__version__} / {torch.version.cuda}")
print_kv("GPU", f"{props.name}, {gpu_total / 2**30:.1f} GiB ({gpu_free / 2**30:.1f} GiB free)")
print_kv("CPU", f"{psutil.cpu_count(logical=False)} cores / {psutil.cpu_count()} threads")
print_kv("RAM", f"{memory['ram_total']:.1f} GiB total, {memory['ram_available']:.1f} GiB free")
print_kv("Commit (RAM + page file)", f"{memory['commit_limit']:.1f} GiB limit, "
                                     f"{memory['commit_available']:.1f} GiB free")
print_kv("Disk free (project drive)", f"{disk.free / 2**30:.0f} GiB")

print_header("Runtime profile (src/config.py)")
print_kv("Precision", str(resolve_amp_dtype(device)).replace("torch.", "") + " autocast"
         + (" + GradScaler" if resolve_amp_dtype(device) == torch.float16 else ""))
print_kv("Batch size / gradient checkpointing", f"{config.DEFAULT_BATCH_SIZE} / "
                                                f"{config.WAV2VEC_GRADIENT_CHECKPOINTING}")
print_kv("Data loading", f"{config.TRAIN_NUM_WORKERS} persistent train worker(s), validation "
                         f"and test in-process, prefetch {config.DATALOADER_PREFETCH_FACTOR}")
print_kv("Feature-store build workers", f"up to {config.FEATURE_STORE_WORKERS}, sized to free memory")
# Measured during a fold: ~10.4 GiB of commit for the training process (the
# GPU memory it reserves is charged to system commit on Windows) + ~2 GiB for
# its worker, and ~3 GiB of RAM; below this, Windows pages and epochs slow down.
if memory["ram_available"] < 4 or memory["commit_available"] < 14:
    print_note(f"Only {memory['ram_available']:.1f} GiB RAM / {memory['commit_available']:.1f} GiB "
               "commit free — close browsers and other heavy applications before training.")

In [ ]:
# -----------------------------------------------------------------------------
# 2.1  Corpus — extracted audio under data/extracted, else extract data/raw/*.tgz
# -----------------------------------------------------------------------------
from src.extraction import extract_all_archives
from src.scanning import scan_audio_files, verify_speakers

EXPECTED_M6 = len(config.ALL_SPEAKERS) * config.WORDS_PER_SPEAKER          # 28 x 765
present_m6 = (sum(1 for _ in config.AUDIO_DIR.rglob(f"*_{config.TARGET_MIC}.wav"))
              if config.AUDIO_DIR.exists() else 0)
print_kv("Audio folder", config.AUDIO_DIR)
if present_m6 >= EXPECTED_M6:
    print_status(f"{present_m6:,} {config.TARGET_MIC} files present — no extraction needed.", ok=True)
else:
    archives = [config.ARCHIVE_DIR / name for name in config.ARCHIVE_FILES]
    missing = [str(a) for a in archives if not a.exists()]
    if missing:
        raise FileNotFoundError(f"Only {present_m6:,}/{EXPECTED_M6:,} M6 files extracted and the "
                                f"archives are missing: {missing}")
    print_kv("Extracting", f"{present_m6:,}/{EXPECTED_M6:,} present — extracting from {config.ARCHIVE_DIR}")
    extract_all_archives()
    verify_speakers(scan_audio_files())

In [ ]:
# -----------------------------------------------------------------------------
# 2.2  Manifest
# -----------------------------------------------------------------------------
df_full = load_manifest()          # outputs/m6_manifest.csv (regenerated from audio if absent)
missing_files = int((~df_full["Filepath"].map(os.path.exists)).sum())
if missing_files:
    raise FileNotFoundError(f"{missing_files:,} manifest files do not exist on disk — "
                            f"delete {config.MANIFEST_PATH} to regenerate it from {config.AUDIO_DIR}.")

print_header("Manifest")
print_kv("Utterances (M6)", f"{len(df_full):,}")
print_kv("Speakers", df_full["Speaker_ID"].nunique())
composition = (df_full.groupby(["Severity", "Block"]).size().unstack(fill_value=0)
               .rename_axis(index="Severity", columns=None).reset_index())
composition["total"] = composition[["B1", "B2", "B3"]].sum(axis=1)
print_table(composition)

In [ ]:
# -----------------------------------------------------------------------------
# 2.3  Run parameters
# -----------------------------------------------------------------------------
RUN_BLOCKS = ["B1", "B2", "B3"]         # the whole corpus: all 765 utterances per speaker
MODEL = SEVERITY_MODEL_NAME             # or an ablation: ab1_wav2vec2_only, ab2_acoustic_only,
                                        # ab3_wav2vec2_acoustic_concat, ab4_wav2vec2_segmental,
                                        # ab5_wav2vec2_suprasegmental, ab6_full_fusion,
                                        # ab7_full_complementarity, ab8_full_speaker_grl
EPOCHS, PATIENCE = 12, 3
BATCH_SIZE = config.DEFAULT_BATCH_SIZE                         # 32 (measured on this machine)
GRADIENT_CHECKPOINTING = config.WAV2VEC_GRADIENT_CHECKPOINTING  # False (measured on this machine)
CACHE_SCOPE = "run"                     # "run": the 15 dysarthric speakers x RUN_BLOCKS; "all": 21,420 files
RUN_BENCHMARK = False                   # True: re-measure batch size x checkpointing (~7 min)

# SMOKE_TEST (or env AAFA_SMOKE=1): 2 folds, 1 epoch, 64 utterances per split,
# under a separate run name — an end-to-end check in ~2 minutes.
SMOKE_TEST = os.environ.get("AAFA_SMOKE") == "1"

RUN_NAME = f"sev_{MODEL}_{'+'.join(RUN_BLOCKS)}_spkval_v2" + ("_smoke" if SMOKE_TEST else "")
df_run = subset_blocks(df_full, RUN_BLOCKS)

# Every fold must be present and complete: all 15 dysarthric speakers, each
# with every utterance of the selected blocks — a missing speaker would
# silently remove a LOSO fold, a short one would bias its test set.
from src.training.runner import build_folds
severity_df = df_run[df_run["Speaker_ID"].isin(config.DYSARTHRIC_IDS)]
per_speaker = severity_df.groupby("Speaker_ID").size()
expected_per_speaker = len(RUN_BLOCKS) * (config.WORDS_PER_SPEAKER // 3)
assert sorted(per_speaker.index) == sorted(config.DYSARTHRIC_IDS), "a dysarthric speaker is missing"
assert (per_speaker == expected_per_speaker).all(), per_speaker[per_speaker != expected_per_speaker]
fold_ids = [fold_id for fold_id, _, _ in build_folds(df_run, "severity", TrainingConfig(task="severity"))]
assert sorted(fold_ids) == sorted(config.DYSARTHRIC_IDS) and len(fold_ids) == 15

print_kv("Run name", RUN_NAME)
print_kv("Blocks", "+".join(RUN_BLOCKS))
print_kv("Severity utterances", f"{len(severity_df):,} = 15 speakers x {expected_per_speaker}")
print_kv("LOSO folds", f"{len(fold_ids)}: {', '.join(fold_ids)}")
print_kv("Class balance (speakers)", ", ".join(
    f"{c} {sum(config.SEVERITY_MAP[s] == c for s in config.DYSARTHRIC_IDS)}"
    for c in config.SEVERITY_CLASS_NAMES))
print_status("All 15 folds present with complete data.", ok=True)
if SMOKE_TEST:
    print_note("SMOKE TEST — pipeline check only, not a result.")

In [ ]:
# -----------------------------------------------------------------------------
# 3.1  Feature store — build only the missing chunks (resumable)
# -----------------------------------------------------------------------------
# One compressed chunk per (speaker, block) under outputs/feature_cache/store,
# each written atomically as it finishes: interrupting this cell loses at most
# the chunks in flight, and re-running it builds only what is missing.
if CACHE_SCOPE == "all":
    df_cache = df_full
else:
    df_cache = df_run[df_run["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].reset_index(drop=True)

feature_store.build_feature_store(df_cache, n_workers=config.FEATURE_STORE_WORKERS)
coverage = feature_store.feature_store_coverage(df_cache)
print_kv("Feature-store coverage", f"{coverage['files_done']:,} / {coverage['files_total']:,} utterances, "
                                   f"{coverage['chunks_done']} / {coverage['chunks_total']} chunks")
if not coverage["complete"]:
    raise RuntimeError(f"Feature store incomplete (missing {coverage['chunks_missing']}) — re-run this cell.")

In [ ]:
# -----------------------------------------------------------------------------
# 3.2  Verification — 300 VAD spans vs live Silero, 24 feature tensors bit-exact
# -----------------------------------------------------------------------------
verify_vad_span_cache(df_cache, n=300)
feature_store.verify_feature_store(df_cache, n=24)

In [ ]:
# -----------------------------------------------------------------------------
# 4.1  One real batch, end to end
# -----------------------------------------------------------------------------
from torch.utils.data import DataLoader
from src.dataset import UASpeechDataset

sample_df = df_cache.sample(n=8, random_state=config.DEFAULT_SEED).reset_index(drop=True)
batch = next(iter(DataLoader(UASpeechDataset(sample_df, include_three_branch=True),
                             batch_size=4, shuffle=False)))
print_header("One batch")
for key in ("waveform", "segmental", "supra", "waveform_length", "supra_valid_frames", "severity_label"):
    print_kv(key, f"{tuple(batch[key].shape)}  {batch[key].dtype}")
assert batch["segmental"].shape[1] == config.SEGMENTAL_CHANNELS
assert batch["supra"].shape[1] == config.SUPRA_CHANNELS
assert batch["waveform"].shape[-1] == config.MAX_SAMPLES
print_status("Batch shapes match the architecture contract.", ok=True)

In [ ]:
# -----------------------------------------------------------------------------
# 4.2  Architecture and parameter counts
# -----------------------------------------------------------------------------
model_preview = build_model(MODEL, config.NUM_CLASSES["severity"], num_speakers=10,
                            gradient_checkpointing=GRADIENT_CHECKPOINTING)
counts = parameter_counts(model_preview)
print_header(f"Model — {MODEL}")
print_kv("Trainable parameters", f"{counts['trainable_params']:,} of {counts['total_params']:,} "
                                 f"({counts['trainable_pct']:.2f}%) — LoRA adapters, branches and heads")
if MODEL == SEVERITY_MODEL_NAME:
    print_feature_audit(model_preview, num_classes=config.NUM_CLASSES["severity"])
del model_preview
torch.cuda.empty_cache()

In [ ]:
# -----------------------------------------------------------------------------
# 5.1  Throughput and projected run time
# -----------------------------------------------------------------------------
if RUN_BENCHMARK:
    from src.training.session import benchmark_batch_sizes
    measurement, benchmark_table = benchmark_batch_sizes(
        df_run, model_name=MODEL, task="severity", batch_sizes=(32, 64))
    BATCH_SIZE, GRADIENT_CHECKPOINTING = measurement.batch_size, measurement.gradient_checkpointing
else:
    # ~1 minute on this machine, with the exact training configuration (batch
    # size, checkpointing, AMP, workers) — so the projection below is measured,
    # not assumed.
    from src.training.session import calibrate_throughput
    measurement = calibrate_throughput(df_run, model_name=MODEL, task="severity",
                                       batch_size=BATCH_SIZE, n_train_samples=640,
                                       n_eval_samples=320,
                                       gradient_checkpointing=GRADIENT_CHECKPOINTING)

projection = project_runtime(df_run, measurement, task="severity", model_name=MODEL, epochs=EPOCHS)
print_header("Projected run time")
print_kv("Batch size / grad. checkpointing", f"{BATCH_SIZE} / {GRADIENT_CHECKPOINTING}")
print_kv("Throughput (train / eval)", f"{measurement.train_samples_per_s:.1f} / "
                                      f"{measurement.eval_samples_per_s:.1f} samples/s")
print_kv("Mean fold size (train/val/test)", f"{projection['n_train']} / {projection['n_val']} / {projection['n_test']}")
print_kv("Epoch", f"{projection['epoch_seconds'] / 60:.1f} min")
print_kv("Fold at max epochs", f"{projection['fold_seconds'] / 60:.1f} min")
print_kv(f"{projection['n_folds']} folds, upper bound", f"{projection['total_seconds'] / 3600:.1f} h "
                                                        f"(early stopping only shortens it)")

In [ ]:
# -----------------------------------------------------------------------------
# 6.1  Frozen configuration
# -----------------------------------------------------------------------------
cfg = TrainingConfig(
    task="severity",
    model=MODEL,
    run_name=RUN_NAME,
    epochs=1 if SMOKE_TEST else EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    num_workers=config.TRAIN_NUM_WORKERS,
    eval_num_workers=config.EVAL_NUM_WORKERS,
    seed=config.DEFAULT_SEED,
    val_protocol="speaker",
    max_folds=2 if SMOKE_TEST else None,
    limit_samples=64 if SMOKE_TEST else None,
)
final_config = print_final_run_configuration(cfg, df_run)
check_frozen_config_guard(final_config)      # refuse a changed config under the same RUN_NAME...
frozen_path = write_frozen_config(final_config)   # ...before recording this one
print_kv("Frozen to", frozen_path.relative_to(PROJECT_ROOT))

In [ ]:
# -----------------------------------------------------------------------------
# 6.2  Training — 15-fold LOSO, resumable
# -----------------------------------------------------------------------------
# Finished folds (metrics + predictions on disk) are loaded, not retrained; a
# fold interrupted mid-way resumes from checkpoints/<RUN_NAME>/<fold>/latest.pt.
# Interrupting this cell (Stop / Ctrl+C) is safe: the current fold stops after
# saving, coverage is printed, and Section 7 reports the folds finished so far.
# Re-run this cell (or Run All) to continue.
training_start = time.monotonic()
torch.cuda.reset_peak_memory_stats()
summary, pooled_metrics = run_training(df_run, cfg)
print()
print_kv("Training wall-clock", f"{(time.monotonic() - training_start) / 3600:.2f} h")
print_kv("Peak GPU memory (this session)", f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

In [ ]:
# -----------------------------------------------------------------------------
# 7.1  Run status and per-fold results  (read from disk — valid after a partial run too)
# -----------------------------------------------------------------------------
from IPython.display import Markdown, display
from src.training.reporting import collect_run_results
from src.training.runner import build_folds

expected_folds = [fold_id for fold_id, _, _ in build_folds(df_run, "severity", cfg)]
if cfg.max_folds:
    expected_folds = expected_folds[:cfg.max_folds]
results = collect_run_results(RUN_NAME, expected_folds)
cov = results["coverage"]

display(Markdown(
    f"### Run status: **{cov['status']}** — {cov['completed']} / {cov['expected']} folds "
    f"({cov['completion_pct']:.1f}%)\n"
    + (f"- **Missing folds:** {', '.join(cov['missing_folds'])}\n" if cov["missing_folds"] else "")
    + (f"- **Failed folds:** {', '.join(cov['failed_folds'])}\n" if cov["failed_folds"] else "")
    + ("\n> **PARTIAL — not a final result.** Re-run the training cell to resume the missing folds.\n"
       if cov["status"] != "COMPLETE" else "")))

per_fold = results["per_fold"]
if per_fold.empty:
    display(Markdown("_No fold has finished yet._"))
else:
    display(per_fold.style
            .format({"Accuracy": "{:.3f}", "Ordinal MAE": "{:.3f}", "Train (min)": "{:.1f}"}, na_rep="N/A")
            .background_gradient(subset=["Accuracy"], cmap="RdYlGn", vmin=0, vmax=1)
            .background_gradient(subset=["Ordinal MAE"], cmap="RdYlGn_r", vmin=0, vmax=3)
            .hide(axis="index")
            .set_caption("Per held-out speaker (each fold has ONE true class, so per-fold "
                         "macro-F1 / balanced accuracy / AUROC are undefined — N/A)"))

In [ ]:
# -----------------------------------------------------------------------------
# 7.2  Pooled evaluation over the completed held-out speakers
# -----------------------------------------------------------------------------
if "pooled" in results:
    tag = "" if cov["status"] == "COMPLETE" else f" — {cov['status']}, NOT A FINAL RESULT"
    display(Markdown(f"### Pooled over {cov['completed']} held-out speakers "
                     f"({results['n_utterances']:,} utterances){tag}"))
    display(results["pooled"].style.format({"Value": "{:.4f}"}, na_rep="N/A")
            .hide(axis="index").set_caption("Headline metrics"))
    display(results["per_class"].rename(columns={
                "class": "Class", "true_n": "True n", "pred_n": "Predicted n", "precision": "Precision",
                "recall": "Recall", "f1": "F1", "auroc_ovr": "AUROC (one-vs-rest)"})
            .style.format({"Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}",
                           "AUROC (one-vs-rest)": "{:.3f}"}, na_rep="N/A")
            .hide(axis="index")
            .set_caption("Per class — N/A: never predicted (precision) or absent from the pooled set"))
    display(results["confusion"].style.background_gradient(cmap="Blues", axis=None)
            .set_caption("Confusion matrix (rows = true, columns = predicted)"))
    display(results["speakers"].style.format({"Utterance accuracy": "{:.3f}"})
            .hide(axis="index")
            .set_caption("Speaker level — median of each speaker's utterance predictions"))
else:
    display(Markdown("_Nothing to pool yet._"))

In [ ]:
# -----------------------------------------------------------------------------
# 7.3  Figures — pooled confusion matrix and per-speaker accuracy
# -----------------------------------------------------------------------------
if "confusion" in results:
    class_names = config.SEVERITY_CLASS_NAMES
    matrix = results["confusion"].to_numpy()
    row_share = matrix / np.clip(matrix.sum(axis=1, keepdims=True), 1, None)
    figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4.6))

    image = left.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
    left.set_xticks(range(4), class_names, rotation=30, ha="right")
    left.set_yticks(range(4), class_names)
    left.set_xlabel("Predicted severity")
    left.set_ylabel("True severity")
    left.set_title(f"Pooled confusion — {cov['completed']}/{cov['expected']} folds, {cov['status']}")
    for i in range(4):
        for j in range(4):
            left.text(j, i, f"{matrix[i, j]:,}\n{row_share[i, j]:.0%}", ha="center", va="center",
                      fontsize=9, color="white" if row_share[i, j] > 0.5 else "black")
    figure.colorbar(image, ax=left, shrink=0.8, label="share of true class")

    ordered = per_fold.assign(order=per_fold["True class"].map(class_names.index)).sort_values(["order", "Fold"])
    colors = plt.cm.viridis(ordered["order"] / 3)
    right.bar(ordered["Fold"], ordered["Accuracy"], color=colors)
    right.set_ylim(0, 1)
    right.set_ylabel("Held-out accuracy")
    right.set_title("Per held-out speaker (colour = true severity)")
    for name, index in zip(class_names, range(4)):
        right.bar(0, 0, color=plt.cm.viridis(index / 3), label=name)
    right.legend(loc="upper right", fontsize=8)
    figure.tight_layout()
    plt.show()

print_kv("Artifacts", config.OUTPUT_DIR)
print_kv("Whole notebook", f"{(time.monotonic() - SESSION_START) / 3600:.2f} h")

## Resuming and ablations

* **Resume**: re-run the notebook (Run All) — or just the training cell after interrupting it. The feature store skips finished chunks; training skips finished folds and resumes an interrupted fold from its last saved epoch.
* **Ablations**: set `MODEL` in 2.3 to one of the `ab*` names and Run All — same data, folds, validation and optimizer; each gets its own `RUN_NAME`. `ab2_acoustic_only` has no wav2vec2 and runs in minutes.
* **Start a configuration over**: delete `outputs/{checkpoints,metrics,predictions}/<RUN_NAME>`.